## Prerequisite

Run `python src/export_model.py` after training to create `run_artifacts/model_package.joblib`. This package is the artifact loaded by the API and Docker image.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd()
src_dir = cwd / 'src' if (cwd / 'src').is_dir() else cwd
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

import pandas as pd
from data import load_joined_data
from score_extractor import FEATURE_NAMES, ScoreExtractor

project_root = src_dir.parent
model_path = project_root / 'run_artifacts' / 'model_package.joblib'
if not model_path.exists():
    raise FileNotFoundError('Run `python src/export_model.py` before executing the scoring cells.')


## Input contract

All fields in `FEATURE_NAMES` are required inside a `features` object. Numeric values must be finite; the six categorical fields must be strings.

In [ ]:
pd.DataFrame({
    'feature': FEATURE_NAMES,
    'type': ['float'] * 19 + ['string'] * 6,
})


## Single-record scoring

The response includes a 0–1 match score, the threshold-based decision, score meaning, model version, record ID, and the container image digest.

In [ ]:
data = load_joined_data().dropna(subset=FEATURE_NAMES)
sample = data.iloc[0]
features = {name: sample[name] for name in FEATURE_NAMES}

scorer = ScoreExtractor(model_path)
single_result = scorer.score_single(features, record_id=str(sample['application_id']))
single_result


## Vectorised batch scoring

Batch requests use the same contract and return the same response structure for every record.

In [ ]:
batch = data.iloc[:5]
batch_results = scorer.score_batch(
    [{name: row[name] for name in FEATURE_NAMES} for _, row in batch.iterrows()],
    [str(record_id) for record_id in batch['application_id']],
)
pd.DataFrame(batch_results)


## REST API

Build and start the container with `docker build -t description-matching-scoring:v1.0.0 .` and `docker run -p 8000:8000 description-matching-scoring:v1.0.0`. Then use `GET /health`, `POST /score`, and `POST /score-batch`; run `python src/edge_case_tests_api.py` for live HTTP validation.